In [2]:
from pytao import TaoModel
from pprint import pprint
import json
import os
from math import sqrt

In [3]:
M = TaoModel('../../../models/cu_linac/tao.init', ploton=False)

Initialized Tao with /var/folders/29/9rt0s28n40xffr24q_0847tm0000gn/T/tmpu9djy_z8/tao/tao.init


In [4]:
from pytao.tao_ctypes.util import parse_bool, parse_tao_lat_ele_list

In [5]:
SLIST = M.cmd_real('python lat_list 1@0>>*|model real:ele.s')
LLIST = M.cmd_real('python lat_list 1@0>>*|model real:ele.l')
NAMES = M.cmd('python lat_list  1@0>>*|model ele.name')

for s, l, n in zip(SLIST[0:20], LLIST[0:20], NAMES[0:20]):
    print (n, l, s)

BEGINNING 0.0 0.0
DL00 -0.8690480000000003 -0.8690480000000003
LOADLOCK 0.8690480000000003 0.0
BEGGUN 0.0 0.0
SOL1BK 0.0 0.0
DBMARK80 0.0 0.0
CATHODE 0.0 0.0
DL01A 0.09600999999999998 0.09600999999999998
SOL1 0.2 0.29601
SOL1#1 0.1 0.19601
YC00 0.0 0.19601
XC00 0.0 0.19601
SQ01 0.0 0.19601
CQ01 0.0 0.19601
SOL1#2 0.1 0.29601
DL01A1 0.07851 0.37451999999999996
VV01 0.0 0.37451999999999996
DL01A2 0.11609 0.49061
AM00 0.0 0.49061
DL01A3 0.10461 0.59522


In [6]:
# Index lookup function
ix_of = parse_tao_lat_ele_list(M.cmd('python lat_ele_list 1@0'))
s_of = {}
for n,s in zip(NAMES, SLIST):
    s_of[n] = s

In [7]:
'K21_4A' in NAMES

True

In [8]:
s_of['K21_4B']

60.4420743449993

In [9]:
CAVS = [name for name in NAMES if name.startswith('K') and ('#' not in name)]
KLYS = {}
for cav in CAVS:
    name, section = cav[0:-1], cav[-1:]
    if name not in KLYS:
        KLYS[name] = []
    KLYS[name].append(section)
KLYS

{'K21_1': ['B', 'C', 'D'],
 'K21_3': ['B', 'C', 'D'],
 'K21_4': ['A', 'B', 'C', 'D'],
 'K21_5': ['A', 'B', 'C', 'D'],
 'K21_6': ['A', 'B', 'C', 'D'],
 'K21_7': ['A', 'B', 'C', 'D'],
 'K21_8': ['A', 'B', 'C', 'D'],
 'K22_1': ['A', 'B', 'C', 'D'],
 'K22_2': ['A', 'B', 'C', 'D'],
 'K22_3': ['A', 'B', 'C', 'D'],
 'K22_4': ['A', 'B', 'C', 'D'],
 'K22_5': ['A', 'B', 'C', 'D'],
 'K22_6': ['A', 'B', 'C', 'D'],
 'K22_7': ['A', 'B', 'C', 'D'],
 'K22_8': ['A', 'B', 'C', 'D'],
 'K23_1': ['A', 'B', 'C', 'D'],
 'K23_2': ['A', 'B', 'C', 'D'],
 'K23_3': ['A', 'B', 'C', 'D'],
 'K23_4': ['A', 'B', 'C', 'D'],
 'K23_5': ['A', 'B', 'C', 'D'],
 'K23_6': ['A', 'B', 'C', 'D'],
 'K23_7': ['A', 'B', 'C', 'D'],
 'K23_8': ['A', 'B', 'C', 'D'],
 'K24_1': ['A', 'B', 'C', 'D'],
 'K24_2': ['A', 'B', 'C', 'D'],
 'K24_3': ['A', 'B', 'C', 'D'],
 'K24_4': ['A', 'B', 'C', 'D'],
 'K24_5': ['A', 'B', 'C', 'D'],
 'K24_6': ['A', 'B', 'C', 'D'],
 'K25_1': ['A', 'B', 'D'],
 'K25_2': ['A', 'B', 'C'],
 'K25_3': ['A', 'B', 'C'],
 

In [10]:
from pytao.util.elements import lat_element
from pytao.util.parameters import tao_parameter_dict

In [11]:
dat = M.cmd('python ele:gen_attribs 1@0>>1491|model')
params = tao_parameter_dict(dat)
params

OrderedDict([('L', REAL;True;2.8692),
             ('units#L', STR;False;m),
             ('TILT', REAL;True;0.0),
             ('units#TILT', STR;False;rad),
             ('RF_FREQUENCY', REAL;True;2856000000.0),
             ('units#RF_FREQUENCY', STR;False;Hz),
             ('GRADIENT', REAL;True;15822375.447498),
             ('units#GRADIENT', STR;False;eV/m),
             ('GRADIENT_ERR', REAL;True;0.0),
             ('units#GRADIENT_ERR', STR;False;eV/m),
             ('VOLTAGE', REAL;False;45397559.633962),
             ('units#VOLTAGE', STR;False;Volt),
             ('VOLTAGE_ERR', REAL;False;0.0),
             ('units#VOLTAGE_ERR', STR;False;Volt),
             ('FRINGE_TYPE', ENUM;True;Full),
             ('FRINGE_AT', ENUM;True;Both_Ends),
             ('SPIN_FRINGE_ON', LOGIC;True;True),
             ('AUTOSCALE_AMPLITUDE', LOGIC;True;True),
             ('AUTOSCALE_PHASE', LOGIC;True;True),
             ('LONGITUDINAL_MODE', INT;True;1),
             ('E_LOSS', REAL;True;

In [12]:
params['GRADIENT'].__class__

pytao.util.parameters.tao_parameter

In [13]:
# Get all parameters for cavities
CAVDAT = {}
EXTRACAVS = ['L1X']
for name in CAVS+EXTRACAVS:
    ix = ix_of[name]
    #print(name, ix)
    dat = M.cmd(f'python ele:gen_attribs 1@0>>{ix}|design')
    params = tao_parameter_dict(dat)
    CAVDAT[name] = params
CAVDAT['K21_1B']['VOLTAGE'].value, CAVDAT['L1X']['VOLTAGE'].value

(45397559.633962, 20000000.0)

In [14]:
{'A', 'B'} == {'B', 'A'}

True

In [15]:

# Klystron power divisions when there is a missing section:
POWER_FACTOR = {
    ("A", "B", "C", "D"): (0.25, 0.25, 0.25, 0.25),
    ("B", "C", "D"): (0.5, 0.25, 0.25),
    ("A", "C", "D"): (0.5, 0.25, 0.25),
    ("A", "B", "C"): (0.25, 0.25, 0.5),
    ("A", "B", "D"): (0.25, 0.25, 0.5),
}
POWER_FACTOR[('A', 'B', 'D')]

(0.25, 0.25, 0.5)

In [16]:
def voltage_factors(sections):
    """
    Voltages go as the sqrt of the power
    """
    s = tuple(sections)
    pfactors = POWER_FACTOR[s]
    vfactors = [sqrt(p) for p in pfactors]
    vtot = sum(vfactors)
    vfactors = [v/vtot for v in vfactors]
    
    return vfactors
voltage_factors(['A', 'B', 'D'])
        

[0.2928932188134525, 0.2928932188134525, 0.4142135623730951]

In [17]:
def klys_data(name, sections):
    """
    
    name: 'K21_1'
    sections = ['B', 'C', 'D']
    
    
    """
    eles = [name+s for s in sections]
    voltages = [CAVDAT[ele]['VOLTAGE'].value for ele in eles]
    phi0s = [CAVDAT[ele]['PHI0'].value for ele in eles]
    gradients = [CAVDAT[ele]['GRADIENT'].value for ele in eles]
    lengths = [CAVDAT[ele]['L'].value for ele in eles]
    
    # Error checking
    assert len(set(phi0s)) == 1, 'phi0 are not unique'
    phi0 = phi0s[0]

    vtot = sum(voltages)
    vfactors = voltage_factors(sections)
    new_gradients = [vf*vtot/ l  for l, vf in zip( lengths, vfactors) ]
    
    
    # Overlay stuff
    
    oname = 'O_'+name
    
    lines = []
    lines.append('!----------------')
    lines.append('! Klystron')
    lines.append('! Configuration: '+''.join(sections))
    lines.append('!')
    lines.append(f'{oname}: overlay = {{')

    for ele, vf, l in zip(eles, vfactors, lengths):
        lines.append(f'    {ele}[gradient]:f*ENLD_MeV*1e6*{vf}/{l},')
 
    for ele, vf, l in zip(eles, vfactors, lengths):
        lines.append(f'    {ele}[gradient_err]:f*ENLD_MeV_err*1e6*{vf}/{l},')

    for ele in eles:
        lines.append(f'    {ele}[phi0]:phase_deg/360,')
        
    for ele in eles:
        lines.append(f'    {ele}[phi0_err]:phase_deg_err/360,')        
        
    lines[-1] = lines[-1][:-1]+'}, var = {ENLD_MeV, ENLD_MeV_err, phase_deg, phase_deg_err, f}, f=1'
    lines.append('\n')
    overlay = '\n'.join(lines)
    
    # Design settings
    # Note that the phase is the local phase. The overall phase will be added by another overlay
    settings = f"""
! Design settings for {name}
{oname}[ENLD_MeV] = {vtot*1e-6}
{oname}[phase_deg] = 0 
"""
    
    #return eles, gradients, new_gradients, overlay
    return {'settings':settings, 'overlay':overlay}
for k,v in KLYS.items():
    print(klys_data(k,v)['overlay'])
    print(klys_data(k,v)['settings'])

!----------------
! Klystron
! Configuration: BCD
!
O_K21_1: overlay = {
    K21_1B[gradient]:f*ENLD_MeV*1e6*0.4142135623730951/2.8692,
    K21_1C[gradient]:f*ENLD_MeV*1e6*0.2928932188134525/2.8692,
    K21_1D[gradient]:f*ENLD_MeV*1e6*0.2928932188134525/3.0441,
    K21_1B[gradient_err]:f*ENLD_MeV_err*1e6*0.4142135623730951/2.8692,
    K21_1C[gradient_err]:f*ENLD_MeV_err*1e6*0.2928932188134525/2.8692,
    K21_1D[gradient_err]:f*ENLD_MeV_err*1e6*0.2928932188134525/3.0441,
    K21_1B[phi0]:phase_deg/360,
    K21_1C[phi0]:phase_deg/360,
    K21_1D[phi0]:phase_deg/360,
    K21_1B[phi0_err]:phase_deg_err/360,
    K21_1C[phi0_err]:phase_deg_err/360,
    K21_1D[phi0_err]:phase_deg_err/360}, var = {ENLD_MeV, ENLD_MeV_err, phase_deg, phase_deg_err, f}, f=1



! Design settings for K21_1
O_K21_1[ENLD_MeV] = 111.55620442640098
O_K21_1[phase_deg] = 0 

!----------------
! Klystron
! Configuration: BCD
!
O_K21_3: overlay = {
    K21_3B[gradient]:f*ENLD_MeV*1e6*0.4142135623730951/3.0441,
    K21_3C[g

In [18]:
CAVDAT['L1X']['VOLTAGE'].value, CAVDAT['L1X']['PHI0'].value*360

(20000000.0, -159.99999999999838)

In [19]:
# Special overlay for L1X

L1X_VOLTAGE = CAVDAT['L1X']['VOLTAGE'].value
L1X_PHI0    = CAVDAT['L1X']['PHI0'].value

L1X_KLYS = f"""
!---------------- 
! Special X-band klystron
O_K21_2: overlay = {{
    L1X[gradient]: f*ENLD_MeV*1e6/L1X[L],
    L1X[gradient_err]: f*ENLD_MeV_err*1e6/L1X[L],
    L1X[phi0]: phase_deg/360,
    L1X[phi0_err]: phase_deg_err/360}},  
    var = {{ ENLD_MeV, ENLD_MeV_err, phase_deg, phase_deg_err, f}}, 
    f=1, ENLD_MeV={L1X_VOLTAGE*1e-6}, phase_deg={round(L1X_PHI0*360,9)}

"""
print(L1X_KLYS)


!---------------- 
! Special X-band klystron
O_K21_2: overlay = {
    L1X[gradient]: f*ENLD_MeV*1e6/L1X[L],
    L1X[gradient_err]: f*ENLD_MeV_err*1e6/L1X[L],
    L1X[phi0]: phase_deg/360,
    L1X[phi0_err]: phase_deg_err/360},  
    var = { ENLD_MeV, ENLD_MeV_err, phase_deg, phase_deg_err, f}, 
    f=1, ENLD_MeV=20.0, phase_deg=-160.0




In [20]:
with open('klystrons.bmad', 'w') as f:

    f.write(L1X_KLYS)
    
    for k,v in KLYS.items():
        f.write(klys_data(k,v)['overlay'])
        
    
        
with open('klystron_design_settings.bmad', 'w') as f:
    for k,v in KLYS.items():
        f.write(klys_data(k,v)['settings'])        

# Linac grouping

In [21]:
s_of['ENDL1'], s_of['ENDL2'], s_of['ENDL3']

(29.465517944109074, 396.0190743449993, 1027.3396600809867)

In [22]:
def linac_of(name):
    s = s_of[name]
    
    if s_of['BEGL1'] <= s <= s_of['ENDL1']:
        return 'L1'
    elif s_of['BEGL2'] <= s <= s_of['ENDL2']:
        return 'L2'    
    elif s_of['BEGL3'] <= s <= s_of['ENDL3']:
        return 'L3'       
    else:
        return None
    

In [23]:
linac_of('K21_1D')

'L1'

In [24]:
# Collect elements
LINAC = {'L1':[],'L2':[],'L3':[]}
for n in CAVS:
    l = linac_of(n)
    LINAC[l].append(n)

In [25]:
LINAC['L1'], len(LINAC['L2']), len(LINAC['L3'])

(['K21_1B', 'K21_1C', 'K21_1D'], 111, 180)

In [26]:
# Find special feedback cavities

L2FEEDBACK = []
L2FORPHASE = []
for n in LINAC['L2']:
    if any([n.startswith(x) for x in ['K24_1', 'K24_2', 'K24_3']]):
        L2FEEDBACK.append(n)
    else:
        L2FORPHASE.append(n)
len(L2FEEDBACK), len(L2FORPHASE)

(12, 99)

In [27]:
L3FEEDBACK = []
L3FORPHASE = []
for n in LINAC['L3']:
    if any([n.startswith(x) for x in ['K29', 'K30']]):
        L3FEEDBACK.append(n)
    else:
        L3FORPHASE.append(n)
len(L3FEEDBACK), len(L3FORPHASE)

(60, 120)

In [28]:
def unique_param(eles, param):
    p = set()
    for ele in eles:
        p.add(CAVDAT[ele][param].value)
    assert len(p) == 1
    return list(p)[0]
L1phi0 = unique_param(LINAC['L1'], 'PHI0')
L2phi0 = unique_param(LINAC['L2'], 'PHI0')
L3phi0 = unique_param(LINAC['L3'], 'PHI0')

In [29]:
L1KLYS = list(set([c[:-1] for c in LINAC['L1']]))
L1KLYS
list(set([c[:-1] for c in LINAC['L2']])).sort()
#list(set([c[:-1] for c in LINAC['L3']]))

In [30]:
# Get klystrons by linac
L1KLYS = list(set([c[:-1] for c in LINAC['L1']]))
L2KLYS = sorted(list(set([c[:-1] for c in LINAC['L2']])))
L3KLYS = sorted(list(set([c[:-1] for c in LINAC['L3']])))
# Klystrons for L2, L3 feedback
L2FEEDBACKKLYS = sorted(list(set([c[:-1] for c in L2FEEDBACK])))
L3FEEDBACKKLYS = sorted(list(set([c[:-1] for c in L3FEEDBACK])))

In [31]:

for name in L2FEEDBACKKLYS:
    print(f'O_{name}[phase_deg] = {round(L2phi0*360,9)}')

O_K24_1[phase_deg] = -33.5
O_K24_2[phase_deg] = -33.5
O_K24_3[phase_deg] = -33.5


In [32]:
# Write to file
with open('linac_phase_defaults.bmad', 'w') as f:
    f.write('! Design linac phasing\n')
    f.write(f'O_L1[phase_deg] = {round(L1phi0*360,9)}\n')
    f.write(f'O_L2[phase_deg] = {round(L2phi0*360,9)}\n')
    f.write(f'O_L3[phase_deg] = {round(L3phi0*360,9)}\n')
    
    f.write('\n!------------------\n')
    f.write('\n! Feedback design phases\n')
    for name in L2FEEDBACKKLYS:
        f.write(f'O_{name}[phase_deg] = {round(L2phi0*360,9)}\n')
        
    for name in L3FEEDBACKKLYS:
        f.write(f'O_{name}[phase_deg] = {round(L3phi0*360,9)}\n')        
    

In [32]:
# Fudge
def fudge_overlays(name, cavs):

    lines = []
    lines.append('!--------------\n')
    lines.append(f'{name}: overlay = {{\n')
    i=0 
    for n in cavs:
        i += 1
        lines.append(f'  O_{n}[f]:f,')
        if i == 4:
            i = 0
            lines.append('\n')   
    if lines[-1] == '\n':
        lines.pop()
        
    lines[-1] = lines[-1][:-1]+'}, var = {f}, f=1\n\n'
    return ''.join(lines)
print(fudge_overlays('O_L1_fudge', L1KLYS))
print(fudge_overlays('O_L2_fudge', L2KLYS))
print(fudge_overlays('O_L3_fudge', L3KLYS))

!--------------
O_L1_fudge: overlay = {
  O_K21_1[f]:f}, var = {f}, f=1


!--------------
O_L2_fudge: overlay = {
  O_K21_3[f]:f,  O_K21_4[f]:f,  O_K21_5[f]:f,  O_K21_6[f]:f,
  O_K21_7[f]:f,  O_K21_8[f]:f,  O_K22_1[f]:f,  O_K22_2[f]:f,
  O_K22_3[f]:f,  O_K22_4[f]:f,  O_K22_5[f]:f,  O_K22_6[f]:f,
  O_K22_7[f]:f,  O_K22_8[f]:f,  O_K23_1[f]:f,  O_K23_2[f]:f,
  O_K23_3[f]:f,  O_K23_4[f]:f,  O_K23_5[f]:f,  O_K23_6[f]:f,
  O_K23_7[f]:f,  O_K23_8[f]:f,  O_K24_1[f]:f,  O_K24_2[f]:f,
  O_K24_3[f]:f,  O_K24_4[f]:f,  O_K24_5[f]:f,  O_K24_6[f]:f}, var = {f}, f=1


!--------------
O_L3_fudge: overlay = {
  O_K25_1[f]:f,  O_K25_2[f]:f,  O_K25_3[f]:f,  O_K25_4[f]:f,
  O_K25_5[f]:f,  O_K25_6[f]:f,  O_K25_7[f]:f,  O_K25_8[f]:f,
  O_K26_1[f]:f,  O_K26_2[f]:f,  O_K26_3[f]:f,  O_K26_4[f]:f,
  O_K26_5[f]:f,  O_K26_6[f]:f,  O_K26_7[f]:f,  O_K26_8[f]:f,
  O_K27_1[f]:f,  O_K27_2[f]:f,  O_K27_3[f]:f,  O_K27_4[f]:f,
  O_K27_5[f]:f,  O_K27_6[f]:f,  O_K27_7[f]:f,  O_K27_8[f]:f,
  O_K28_1[f]:f,  O_K28_2[f]:f,  O_K

In [33]:
# Overall phase overlays
with open('linac_fudge_overlays.bmad', 'w') as f:
    f.write(fudge_overlays('O_L1_fudge', L1KLYS))
    f.write(fudge_overlays('O_L2_fudge', L2KLYS))
    f.write(fudge_overlays('O_L3_fudge', L3KLYS))

In [34]:
# Make overlays
def phase_overlay(name, cavs):
    lines = []
    lines.append('!--------------\n')
    lines.append('! Linac phase overlay\n')
    lines.append(f'{name}: overlay = {{\n')
    i=0
    for n in cavs:
        i += 1
        lines.append(f'  {n}[phi0]:phase_deg/360,')
        if i == 4:
            i = 0
            lines.append('\n')   
    if lines[-1] == '\n':
        lines.pop()
        
    lines[-1] = lines[-1][:-1]+'}, var = {phase_deg}\n\n'
    return ''.join(lines)
print(phase_overlay('O_L1', LINAC['L1'])           )
print(phase_overlay('O_L2', L2FORPHASE)           )
print(phase_overlay('O_L3', L3FORPHASE)           )

!--------------
! Linac phase overlay
O_L1: overlay = {
  K21_1B[phi0]:phase_deg/360,  K21_1C[phi0]:phase_deg/360,  K21_1D[phi0]:phase_deg/360}, var = {phase_deg}


!--------------
! Linac phase overlay
O_L2: overlay = {
  K21_3B[phi0]:phase_deg/360,  K21_3C[phi0]:phase_deg/360,  K21_3D[phi0]:phase_deg/360,  K21_4A[phi0]:phase_deg/360,
  K21_4B[phi0]:phase_deg/360,  K21_4C[phi0]:phase_deg/360,  K21_4D[phi0]:phase_deg/360,  K21_5A[phi0]:phase_deg/360,
  K21_5B[phi0]:phase_deg/360,  K21_5C[phi0]:phase_deg/360,  K21_5D[phi0]:phase_deg/360,  K21_6A[phi0]:phase_deg/360,
  K21_6B[phi0]:phase_deg/360,  K21_6C[phi0]:phase_deg/360,  K21_6D[phi0]:phase_deg/360,  K21_7A[phi0]:phase_deg/360,
  K21_7B[phi0]:phase_deg/360,  K21_7C[phi0]:phase_deg/360,  K21_7D[phi0]:phase_deg/360,  K21_8A[phi0]:phase_deg/360,
  K21_8B[phi0]:phase_deg/360,  K21_8C[phi0]:phase_deg/360,  K21_8D[phi0]:phase_deg/360,  K22_1A[phi0]:phase_deg/360,
  K22_1B[phi0]:phase_deg/360,  K22_1C[phi0]:phase_deg/360,  K22_1D[phi0]:phas

In [37]:
# Make Subbooster overlays
def sbst_overlay(sector, stations):
    lines = []
    lines.append('!--------------\n')
    lines.append(f'!subbooster {sector} overlay\n')
    lines.append(f'O_SBST_{sector}: overlay = {{\n')
    i=0
    for n in stations:
        i += 1
        lines.append(f'  O_K{sector}_{n}[phase_deg]:sbst_phase_deg,\n')
    if lines[-1] == '\n':
        lines.pop()
    lines[-1] = lines[-1][:-1]+'}, var = {sbst_phase_deg}\n\n'
    return ''.join(lines)
print(sbst_overlay('22', (1,2,3,4,5,6,7,8)))

!--------------
!subbooster 22 overlay
O_SBST_22: overlay = {
  O_K22_1[phase_deg]:sbst_phase_deg,
  O_K22_2[phase_deg]:sbst_phase_deg,
  O_K22_3[phase_deg]:sbst_phase_deg,
  O_K22_4[phase_deg]:sbst_phase_deg,
  O_K22_5[phase_deg]:sbst_phase_deg,
  O_K22_6[phase_deg]:sbst_phase_deg,
  O_K22_7[phase_deg]:sbst_phase_deg,
  O_K22_8[phase_deg]:sbst_phase_deg,}, var = {sbst_phase_deg}




# Write to file

In [35]:
L3FEEDBACK

['K29_1A',
 'K29_1B',
 'K29_1C',
 'K29_2A',
 'K29_2B',
 'K29_2C',
 'K29_2D',
 'K29_3A',
 'K29_3B',
 'K29_3C',
 'K29_3D',
 'K29_4A',
 'K29_4B',
 'K29_4C',
 'K29_5A',
 'K29_5B',
 'K29_5C',
 'K29_6A',
 'K29_6B',
 'K29_6C',
 'K29_6D',
 'K29_7A',
 'K29_7B',
 'K29_7C',
 'K29_7D',
 'K29_8A',
 'K29_8B',
 'K29_8C',
 'K29_8D',
 'K30_1A',
 'K30_1B',
 'K30_1C',
 'K30_1D',
 'K30_2A',
 'K30_2B',
 'K30_2C',
 'K30_2D',
 'K30_3A',
 'K30_3B',
 'K30_3C',
 'K30_3D',
 'K30_4A',
 'K30_4B',
 'K30_4C',
 'K30_4D',
 'K30_5A',
 'K30_5B',
 'K30_5C',
 'K30_5D',
 'K30_6A',
 'K30_6B',
 'K30_6C',
 'K30_6D',
 'K30_7A',
 'K30_7B',
 'K30_7C',
 'K30_7D',
 'K30_8A',
 'K30_8B',
 'K30_8C']

In [36]:
# Overall phase overlays
with open('linac_phase_overlays.bmad', 'w') as f:
    f.write( phase_overlay('O_L1', LINAC['L1']) )
    f.write(phase_overlay('O_L2', L2FORPHASE)     )
    f.write(phase_overlay('O_L3', L3FORPHASE)     )

In [37]:
!cat linac_phase_overlays.bmad

!--------------
! Linac phase overlay
O_L1: overlay = {
  K21_1B[phi0]:phase_deg/360,  K21_1C[phi0]:phase_deg/360,  K21_1D[phi0]:phase_deg/360}, var = {phase_deg}

!--------------
! Linac phase overlay
O_L2: overlay = {
  K21_3B[phi0]:phase_deg/360,  K21_3C[phi0]:phase_deg/360,  K21_3D[phi0]:phase_deg/360,  K21_4A[phi0]:phase_deg/360,
  K21_4B[phi0]:phase_deg/360,  K21_4C[phi0]:phase_deg/360,  K21_4D[phi0]:phase_deg/360,  K21_5A[phi0]:phase_deg/360,
  K21_5B[phi0]:phase_deg/360,  K21_5C[phi0]:phase_deg/360,  K21_5D[phi0]:phase_deg/360,  K21_6A[phi0]:phase_deg/360,
  K21_6B[phi0]:phase_deg/360,  K21_6C[phi0]:phase_deg/360,  K21_6D[phi0]:phase_deg/360,  K21_7A[phi0]:phase_deg/360,
  K21_7B[phi0]:phase_deg/360,  K21_7C[phi0]:phase_deg/360,  K21_7D[phi0]:phase_deg/360,  K21_8A[phi0]:phase_deg/360,
  K21_8B[phi0]:phase_deg/360,  K21_8C[phi0]:phase_deg/360,  K21_8D[phi0]:phase_deg/360,  K22_1A[phi0]:phase_deg/360,
  K22_1B[phi0]:phase_deg/360,  K22_1C[phi0]:phase_deg/360,  K22_1D[phi0]:phase

In [40]:
# Subbooster overlays
with open('sbst_phase_overlays.bmad', 'w') as f:
    f.write(sbst_overlay('21', (3,4,5,6,7,8))) #21-1 (L1S) and 21-2 (L1X) dont use the subbooster.
    f.write(sbst_overlay('22', (1,2,3,4,5,6,7,8)))
    f.write(sbst_overlay('23', (1,2,3,4,5,6,7,8)))
    f.write(sbst_overlay('24', (4,5,6))) #1,2,3 are feedback stations, no sbst.  7 is a tcav, 8 doesn't exist.
    f.write(sbst_overlay('25', (1,2,3,4,5,6,7,8)))
    f.write(sbst_overlay('26', (1,2,3,4,5,6,7,8)))
    f.write(sbst_overlay('27', (1,2,3,4,5,6,7,8)))
    f.write(sbst_overlay('28', (1,2,3,4,5,6,7,8)))
    f.write(sbst_overlay('29', (1,2,3,4,5,6,7,8)))
    f.write(sbst_overlay('30', (1,2,3,4,5,6,7,8)))